# Voice2Reel — استنساخ الصوت العربي وتخصيص XTTS-v2
اختر **T4 GPU** وشغّل هذا الدفتر في جلسة مستقلة عن تدريب السكربت.

يمكن تجربة استنساخ الصوت بالنموذج الأساسي قبل أي تدريب. التخصيص يحتاج صوتا حقيقيا ونصا مطابقا لكل ملف.
تدرب الوصفة جزء GPT من XTTS فقط. لا توجد أوزان جديدة مدرّبة ضمن هذا المستودع.

اقرأ [ترخيص أوزان XTTS](https://huggingface.co/coqui/XTTS-v2/blob/main/LICENSE.txt): CPML، والاستخدام التجاري ليس مشمولا تلقائيا.
حزمة التسجيل في `datasets/voice/recording_prompts.csv` تحتوي 120 جملة لتسجيلها، وليست مقاطع صوتية جاهزة.
استهدف 20–60 دقيقة نظيفة أو أكثر للتخصيص الجاد؛ الجمل المرفقة تكفي لبداية وتجربة المسار فقط.

ارفع ZIP للمستودع من GitHub في الخلية التالية.

In [ ]:
from pathlib import Path
import zipfile
import os
import subprocess
import sys
from google.colab import files, drive

# Download Code -> Download ZIP from your private GitHub repository, then upload it here.
if not Path('/content/Voice2Reel/pyproject.toml').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one repository ZIP from GitHub')
    destination = Path('/content/repo_upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(info.file_size for info in archive.infolist()) > 50_000_000:
            raise ValueError('Expected source-only repository ZIP, under 50 MB')
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    projects = list(destination.rglob('pyproject.toml'))
    if len(projects) != 1:
        raise ValueError('Cannot identify repository root')
    import shutil
    shutil.copytree(projects[0].parent, '/content/Voice2Reel')
    del uploaded
os.chdir('/content/Voice2Reel')
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/Voice2Reel')
WORK.mkdir(parents=True, exist_ok=True)
print('Persistent artifacts:', WORK)


## التثبيت

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.8'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/content/v2r-env'], check=True)
PY = '/content/v2r-env/bin/python'
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args], check=True)
def run(*args):
    subprocess.run([PY, *map(str, args)], check=True)
pip_install('torch==2.8.0', 'torchaudio==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126')
pip_install('-r', 'requirements/voice.txt')
run('-c', "import torch; assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'; print(torch.cuda.get_device_name(0)); print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)")
# Isolated Python environment: no kernel restart or changes to Colab's preinstalled packages.

freeze = subprocess.check_output([sys.executable, "-m", "uv", "pip", "freeze", "--python", PY], text=True)
(WORK / "02_arabic_voice_xtts_environment.txt").write_text(freeze)


## عينة مرجعية
ضع عينة WAV نظيفة مدتها 3–30 ثانية في المسار التالي على Drive. هذا المرجع للتجربة، لا يُستخدم كبيانات تدريب كاملة.

In [ ]:
REFERENCE = WORK / 'reference.wav'
if not REFERENCE.is_file():
    uploaded = files.upload()
    wavs = [name for name in uploaded if name.lower().endswith('.wav')]
    if len(wavs) != 1:
        raise ValueError('Upload one reference WAV')
    REFERENCE.write_bytes(uploaded[wavs[0]])
    del uploaded
run('-c', f"from voice2reel.voice.prepare import load_audio; load_audio({str(REFERENCE)!r}, 3, 30); print('Reference OK')")


## تجربة الاستنساخ قبل التدريب
استخدم نفس المرجع والنص عند المقارنة مع النموذج المخصص.

In [ ]:
TEST_TEXT = WORK / 'voice_test.txt'
TEST_TEXT.write_text('هل تستطيع شرح فكرتك في جملة واحدة، ثم دعمها بمثال واضح؟', encoding='utf-8')
run('-m', 'voice2reel.voice.synthesize', '--reference', REFERENCE, '--text', TEST_TEXT,
    '--output', WORK / 'baseline.wav')
from IPython.display import Audio, display
display(Audio(str(WORK / 'baseline.wav')))


## تجهيز تسجيلات التدريب
على Drive أنشئ `Voice2Reel/recordings/` وضع ملفات WAV داخله. انسخ `manifest.template.jsonl` إلى `Voice2Reel/manifest.jsonl` واحذف الأسطر التي لم تسجلها.
كل سطر يربط اسم ملف بنصه الحقيقي. لا تغير النص دون إعادة تسجيله أو تصحيحه وفق الصوت.
المقاطع بين 2 و11 ثانية، لمتحدث واحد، بدون موسيقى. الحد الأدنى التقني 12 مقطعا؛ الجودة تحتاج أكثر.

إذا كنت تريد تجربة الاستنساخ فقط يمكنك التوقف هنا؛ الخلايا التالية تتطلب تسجيلاتك.

In [ ]:
RAW_AUDIO = WORK / 'recordings'
MANIFEST = WORK / 'manifest.jsonl'
PREPARED = WORK / 'voice_dataset_01'
if not PREPARED.exists():
    run('-m', 'voice2reel.voice.prepare', '--manifest', MANIFEST, '--audio-root', RAW_AUDIO,
        '--output', PREPARED)
print((PREPARED / 'report.json').read_text()[:1500])


## تخصيص الصوت
ابدأ بعصر واحد للتأكد من عمل التدريب على GPU، واسمع النتيجة. استخدم مجلد تجربة جديدا لتغيير عدد العصور.
RESUME هو مجلد تشغيل Coqui الذي يحتوي `config.json` وcheckpoint، وليس ملف أوزان منفردا.
عند الاستئناف تعود إعدادات التشغيل الأصلية؛ لا تغير موقع البيانات المحضرة.

In [ ]:
VOICE_RUNS = WORK / 'voice_runs'
RESUME = ''
EPOCHS = 1  # Smoke run first; try 5 in a new run after checking audio quality.
command = ['-m', 'voice2reel.voice.train', '--data', PREPARED, '--output', VOICE_RUNS,
           '--epochs', EPOCHS, '--grad-accum', '16']
if RESUME:
    command += ['--resume', RESUME]
run(*command)


## اختيار checkpoint ومقارنة النتيجة
اختر أفضل checkpoint بحسب validation loss من التجربة التي تقيمها؛ قيمة loss ليست مقياس تشابه صوتي.

In [ ]:
candidates = sorted(VOICE_RUNS.rglob('best_model.pth'), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError('No best_model.pth; check training log and checkpoints')
CHECKPOINT = candidates[-1]  # Change explicitly if evaluating a different run.
print('Selected:', CHECKPOINT)
run('-m', 'voice2reel.voice.synthesize', '--reference', REFERENCE, '--text', TEST_TEXT,
    '--checkpoint', CHECKPOINT, '--output', WORK / 'finetuned.wav')
display(Audio(str(WORK / 'finetuned.wav')))


## قراءة سكربت النموذج الأول
إذا شغلت دفتر السكربت، سيجد هذه الخلية `script.json` على Drive.
يُولد كل مقطع منفردا وتُحفظ بدايته ونهايته الحقيقية بجانب WAV. هذه توقيتات مقاطع وليست توقيتات كلمات أو فيديو مرندرا.

In [ ]:
SCRIPT = WORK / 'script.json'
if SCRIPT.exists():
    run('-m', 'voice2reel.voice.synthesize', '--reference', REFERENCE, '--script', SCRIPT,
        '--checkpoint', CHECKPOINT, '--output', WORK / 'narration.wav')
    display(Audio(str(WORK / 'narration.wav')))
else:
    print('Run the scriptwriter notebook to create script.json, then return here.')


## الملفات الناتجة
الأوزان محفوظة بالفعل على Drive داخل مجلد التشغيل، ويمكن تنزيلها من هناك.
لإعادة الاستخدام احتفظ بـ`best_model.pth`، و`voice2reel_run.json`، وملفات النموذج الأساسي المحددة في الكود.
قارن الجمل غير المستخدمة في التدريب، وسجل أخطاء النطق وحذف الكلمات والتكرار وتشابه الصوت.
انظر `docs/EVALUATION.md` لحساب WER/CER من تفريغ الصوت ومراجعة النتائج.